# Tripwire — team setup guide (T11)

Companion to `docs/setup.md`. Packages Manahil's real, working T01 setup
(`scripts/t01_sanity_test.py`, merged via PR #2) into a notebook the whole team can run — this
clones and runs her actual script, it doesn't reimplement it.

**Platform notes, confirmed not guessed:**
- Native Windows doesn't work: `textworld`'s dependency `jericho` has no Windows wheel and
  fails to build from source (`FileNotFoundError: [WinError 2]`, it shells out to a Unix build
  tool). Use Colab or WSL2/Linux.
- Colab's own default Python (3.13) doesn't work either: `alfworld.agents.environment`
  crashes on `env.reset()` with `NameError: name 'r' is not defined` inside
  `textworld/envs/pddl/textgen/__init__.py` (upstream TextWorld bug on very new Python).
  Fixed by installing into an isolated venv pinned to Python 3.10 (matching Manahil's setup)
  and running everything through that venv's interpreter.

**Model, not Llama-3.1-8B-Instruct:** CLAUDE.md fixes the agent as Llama-3.1-8B-Instruct, but
that's currently unreachable on Groq (`404 model_not_found`; no Llama model at all on a tested
key's `/v1/models` list). `openai/gpt-oss-20b` was tried next and rejected — it ships
built-in tools that fire with no way to disable them, breaking a plain ReAct loop. Manahil's
PR independently landed on **`qwen/qwen3.8-27b`**, which this notebook uses — still flagged to
the team as a real change to a documented assumption, not something either of us decided alone.

## 1. Clone the team repo

In [ ]:
!git clone https://github.com/ManahilAyaz/tripwire-agent-monitor.git
%cd tripwire-agent-monitor

## 2. Isolated venv (Python 3.10) and dependencies

Everything ALFWorld-related runs through `/content/venv/bin/python` from here on, never
Colab's own kernel Python — that's what avoids the `textgen` `NameError`.

In [ ]:
!pip install -q uv
!test -d /content/venv || uv venv --python 3.10 /content/venv
!uv pip install -q --python /content/venv/bin/python alfworld openai pyyaml

## 3. Download game data and fetch the base config

In [ ]:
import os

os.environ["ALFWORLD_DATA"] = "/content/alfworld_data"
!/content/venv/bin/alfworld-download
!git clone https://github.com/alfworld/alfworld
!cp alfworld/configs/base_config.yaml .

## 4. Set the Groq API key

Entered through a secure prompt, never hard-coded. Don't reuse any key that's ever been pasted
into a notebook or chat in plaintext — rotate it first.

In [ ]:
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("GROQ_API_KEY: ")

## 5. Run the real T01 sanity script

In [ ]:
!/content/venv/bin/python scripts/t01_sanity_test.py --model qwen/qwen3.8-27b --steps 5 --out notebooks/t01_sample_output.json

import json
from pathlib import Path

output_file = Path("notebooks/t01_sample_output.json")
assert output_file.exists(), "Output JSON was not generated."

with open(output_file) as f:
    result = json.load(f)

print("Output JSON verified successfully!")
print("Number of recorded steps:", len(result.get("steps", [])))

## 6. Where T12 picks up

This script dumps one whole-episode JSON file. T12's job is reshaping that into one JSONL line
per step plus an episode-summary line, once `docs/log_schema.md` (T03) is agreed — see the
schema draft proposal (shared with Manahil directly, not part of this repo), which already
uses this script's real field names (`thought`, `action`, `observation`) as its basis. That's
separate work, in `src/tripwire/trajectory/` — not something this setup notebook needs to do.